In [14]:
import os
import json
import numpy as np
import random
import numpy as np
from skimage.io import imread
from skimage.filters import threshold_otsu

ANNOTATOR_1_DIR = "../annotations/2_annotators_CD4_annotator_1/"
ANNOTATOR_2_DIR = "../annotations/2_annotators_CD4_annotator_2/"

In [20]:
def load_annotations(directory):
    annotations = {}
    for fname in os.listdir(directory):
        if fname.endswith('.json'):
            with open(os.path.join(directory, fname)) as f:
                data = json.load(f)
                slide_name = data['tile']
                # Assumes JSON has 'min_range' and 'max_range' keys
                annotations[slide_name] = {'min': data['min'], 'max': data['max']}
    return annotations

def compute_errors(ann1, ann2):
    min_errors = []
    max_errors = []
    for fname in ann1:
        if fname in ann2:
            min1, max1 = ann1[fname]['min'], ann1[fname]['max']
            min2, max2 = ann2[fname]['min'], ann2[fname]['max']
            min_errors.append(abs(min1 - min2))
            max_errors.append(abs(max1 - max2))
    return np.array(min_errors), np.array(max_errors)

def random_distribution(ann1, ann2):
    # Shuffle min/max independently to simulate random pairing
    min1 = [v['min'] for v in ann1.values()]
    max1 = [v['max'] for v in ann1.values()]
    min2 = [v['min'] for v in ann2.values()]
    max2 = [v['max'] for v in ann2.values()]
    random.shuffle(min2)
    random.shuffle(max2)
    min_errors = np.abs(np.array(min1) - np.array(min2))
    max_errors = np.abs(np.array(max1) - np.array(max2))
    return min_errors, max_errors

def percentile_range(image_path, min_pct=1, max_pct=99):
    img = imread(image_path)
    min_val = np.percentile(img, min_pct)
    max_val = np.percentile(img, max_pct)
    return min_val, max_val

    from skimage.filters import threshold_otsu

def otsu_range(image_path):
    img = imread(image_path)
    thresh = threshold_otsu(img)
    return img.min(), thresh  # or thresh, img.max()

def compute_baseline_errors(ann, method='otsu', percentile=(1,99)):
    min_errors = []
    max_errors = []
    for slide_name in ann:
        # Assume image path can be constructed from slide_name
        # Update this path logic as needed for your setup
        image_path = f"../data/2_annotators_CD4/tiles_marker/{slide_name}"
        try:
            if method == 'otsu':
                img = imread(image_path)
                thresh = threshold_otsu(img)
                min_val, max_val = img.min(), thresh
            elif method == 'percentile':
                min_val, max_val = percentile_range(image_path, min_pct=percentile[0], max_pct=percentile[1])
            else:
                continue
            min_errors.append(abs(ann[slide_name]['min'] - min_val))
            max_errors.append(abs(ann[slide_name]['max'] - max_val))
        except Exception as e:
            print(f"Error processing {slide_name}: {e}")
    return np.array(min_errors), np.array(max_errors)

In [18]:
ann1 = load_annotations(ANNOTATOR_1_DIR)
ann2 = load_annotations(ANNOTATOR_2_DIR)

In [32]:
min_errors, max_errors = compute_errors(ann1, ann2)
rand_min_errors, rand_max_errors = random_distribution(ann1, ann2)
print("Annotator min error mean:", np.mean(min_errors))
print("Random min error mean:", np.mean(rand_min_errors))
print("Annotator max error mean:", np.mean(max_errors))
print("Random max error mean:", np.mean(rand_max_errors))

Annotator min error mean: 0.04777874678840639
Random min error mean: 0.2944455221924453
Annotator max error mean: 0.09192181807776845
Random max error mean: 0.327919004202406


In [21]:
# Otsu baseline
otsu_min_errors_1, otsu_max_errors_1 = compute_baseline_errors(ann1, method='otsu')
otsu_min_errors_2, otsu_max_errors_2 = compute_baseline_errors(ann2, method='otsu')

# Percentile baseline
perc_min_errors_1, perc_max_errors_1 = compute_baseline_errors(ann1, method='percentile', percentile=(1,99))
perc_min_errors_2, perc_max_errors_2 = compute_baseline_errors(ann2, method='percentile', percentile=(1,99))

print("Annotator 1 vs Otsu min error mean:", np.mean(otsu_min_errors_1))
print("Annotator 2 vs Otsu min error mean:", np.mean(otsu_min_errors_2))
print("Annotator 1 vs Otsu max error mean:", np.mean(otsu_max_errors_1))
print("Annotator 2 vs Otsu max error mean:", np.mean(otsu_max_errors_2))

print("Annotator 1 vs Percentile min error mean:", np.mean(perc_min_errors_1))
print("Annotator 2 vs Percentile min error mean:", np.mean(perc_min_errors_2))
print("Annotator 1 vs Percentile max error mean:", np.mean(perc_max_errors_1))
print("Annotator 2 vs Percentile max error mean:", np.mean(perc_max_errors_2))

Annotator 1 vs Otsu min error mean: 0.49639517521622734
Annotator 2 vs Otsu min error mean: 0.490178602077223
Annotator 1 vs Otsu max error mean: 0.24845512255480076
Annotator 2 vs Otsu max error mean: 0.3113662784136279
Annotator 1 vs Percentile min error mean: 0.19734748350699147
Annotator 2 vs Percentile min error mean: 0.19113947841716325
Annotator 1 vs Percentile max error mean: 0.0908158914704819
Annotator 2 vs Percentile max error mean: 0.1460641318595828


In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

In [23]:
# Prepare data for plotting
mean_errors = (min_errors + max_errors) / 2
rand_mean_errors = (rand_min_errors + rand_max_errors) / 2

plot_data = {
    "Inter-annotator": {
        "min": min_errors,
        "max": max_errors,
        "mean": mean_errors
    },
    "Random": {
        "min": rand_min_errors,
        "max": rand_max_errors,
        "mean": rand_mean_errors
    }
}

In [25]:
import matplotlib.backends.backend_pdf

pdf_path = "../annotations/interannotator_vs_random_CD4_100_tiles.pdf"
pdf = matplotlib.backends.backend_pdf.PdfPages(pdf_path)

def plot_violin_jitter_save(data1, data2, label1, label2, title, ylabel, pdf):
    fig, ax = plt.subplots()
    all_data = [data1, data2]
    labels = [label1, label2]
    sns.violinplot(data=all_data, inner=None, palette=["#4C72B0", "#DD8452"], ax=ax)
    sns.stripplot(data=all_data, color='k', alpha=0.4, jitter=0.25, ax=ax)
    t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
    ax.text(0.5, max(np.max(data1), np.max(data2)) * 0.95, f"t-test p = {p_val:.2e}", 
            ha='center', va='top', fontsize=14, color='black')
    ax.set_xticklabels(labels)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    sns.despine()
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

plot_violin_jitter_save(
    plot_data["Inter-annotator"]["min"], 
    plot_data["Random"]["min"], 
    "Inter-annotator", "Random", 
    "Error Distribution (Min Range)", 
    "Absolute Error",
    pdf
)
plot_violin_jitter_save(
    plot_data["Inter-annotator"]["max"], 
    plot_data["Random"]["max"], 
    "Inter-annotator", "Random", 
    "Error Distribution (Max Range)", 
    "Absolute Error",
    pdf
)
plot_violin_jitter_save(
    plot_data["Inter-annotator"]["mean"], 
    plot_data["Random"]["mean"], 
    "Inter-annotator", "Random", 
    "Error Distribution (Mean of Min & Max)", 
    "Absolute Error",
    pdf
)

pdf.close()
print(f"Saved to {pdf_path}")

/tmp/ipykernel_7264/3369167308.py:15: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels)
/tmp/ipykernel_7264/3369167308.py:15: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels)


Saved to ../annotations/interannotator_vs_random_CD4_100_tiles.pdf


/tmp/ipykernel_7264/3369167308.py:15: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels)


In [26]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import matplotlib.backends.backend_pdf

# Prepare data for scatter plots
common_slides = sorted(set(ann1.keys()) & set(ann2.keys()))
ann1_min = np.array([ann1[k]['min'] for k in common_slides])
ann2_min = np.array([ann2[k]['min'] for k in common_slides])
ann1_max = np.array([ann1[k]['max'] for k in common_slides])
ann2_max = np.array([ann2[k]['max'] for k in common_slides])

pdf_path_corr = "../annotations/interannotator_correlation_CD4_100_tiles.pdf"
pdf_corr = matplotlib.backends.backend_pdf.PdfPages(pdf_path_corr)

def plot_scatter_with_corr(x, y, xlabel, ylabel, title, pdf):
    fig, ax = plt.subplots(figsize=(7,7))
    sns.scatterplot(x=x, y=y, ax=ax, color="#4C72B0", s=60, edgecolor='k', alpha=0.8)
    # Diagonal line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
        np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
    ]
    ax.plot(lims, lims, '--', color='gray', linewidth=1)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    # Pearson correlation
    r, p = pearsonr(x, y)
    ax.text(0.05, 0.95, f"Pearson r = {r:.2f}\np = {p:.2e}", 
            transform=ax.transAxes, ha='left', va='top', fontsize=15, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    sns.despine()
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

# Plot for min
plot_scatter_with_corr(
    ann1_min, ann2_min,
    "Annotator 1 Min", "Annotator 2 Min",
    "Min Range: Annotator 1 vs Annotator 2",
    pdf_corr
)

# Plot for max
plot_scatter_with_corr(
    ann1_max, ann2_max,
    "Annotator 1 Max", "Annotator 2 Max",
    "Max Range: Annotator 1 vs Annotator 2",
    pdf_corr
)

pdf_corr.close()
print(f"Saved to {pdf_path_corr}")

Saved to ../annotations/interannotator_correlation_CD4_100_tiles.pdf


In [31]:
# Find top 5 tiles with largest difference in min and max range

# Compute differences for all common slides
common_slides = sorted(set(ann1.keys()) & set(ann2.keys()))
min_diffs = {k: abs(ann1[k]['min'] - ann2[k]['min']) for k in common_slides}
max_diffs = {k: abs(ann1[k]['max'] - ann2[k]['max']) for k in common_slides}

# Sort and get top 5
top5_min = sorted(min_diffs.items(), key=lambda x: x[1], reverse=True)[:5]
top5_max = sorted(max_diffs.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 tiles with largest min range difference:")
for tile, diff in top5_min:
    print(
        f"{tile}: {diff:.5f} "
        f"(annotator 1 min = {ann1[tile]['min']:.5f}, max = {ann1[tile]['max']:.5f} / "
        f"annotator 2 min = {ann2[tile]['min']:.5f}, max = {ann2[tile]['max']:.5f})"
    )
print("Annotator 1 min values:", [f"{ann1[tile]['min']:.5f}" for tile, _ in top5_min])
print("Annotator 2 min values:", [f"{ann2[tile]['min']:.5f}" for tile, _ in top5_min])

print("\nTop 5 tiles with largest max range difference:")
for tile, diff in top5_max:
    print(
        f"{tile}: {diff:.5f} "
        f"(annotator 1 min = {ann1[tile]['min']:.5f}, max = {ann1[tile]['max']:.5f} / "
        f"annotator 2 min = {ann2[tile]['min']:.5f}, max = {ann2[tile]['max']:.5f})"
    )
print("Annotator 1 max values:", [f"{ann1[tile]['max']:.5f}" for tile, _ in top5_max])
print("Annotator 2 max values:", [f"{ann2[tile]['max']:.5f}" for tile, _ in top5_max])

Top 5 tiles with largest min range difference:
MF066_early_TMA1_x480_y1348_CD4.tiff: 0.62258 (annotator 1 min = 0.00509, max = 0.01062 / annotator 2 min = 0.62767, max = 0.83941)
MF197_late_2_TMA1_x3456_y1503_CD4.tiff: 0.47232 (annotator 1 min = 0.02705, max = 0.04296 / annotator 2 min = 0.49937, max = 0.69592)
MF035_late_TMA1_x4819_y2268_CD4.tiff: 0.20678 (annotator 1 min = 0.25070, max = 1.00000 / annotator 2 min = 0.45748, max = 0.94787)
AD038_TMA1_x1558_y2196_CD4.tiff: 0.16735 (annotator 1 min = 0.41735, max = 0.76421 / annotator 2 min = 0.25000, max = 1.00000)
MF021_early_2_TMA2_x6246_y661_CD4.tiff: 0.15054 (annotator 1 min = 0.00455, max = 0.37866 / annotator 2 min = 0.15509, max = 0.37866)
Annotator 1 min values: ['0.00509', '0.02705', '0.25070', '0.41735', '0.00455']
Annotator 2 min values: ['0.62767', '0.49937', '0.45748', '0.25000', '0.15509']

Top 5 tiles with largest max range difference:
MF066_early_TMA1_x480_y1348_CD4.tiff: 0.82879 (annotator 1 min = 0.00509, max = 0.0106

In [33]:
# Remove top 5 slides with largest min and max differences and recompute inter-annotator errors

# Get the set of slides to remove (union of both top5 lists)
slides_to_remove = set([tile for tile, _ in top5_min] + [tile for tile, _ in top5_max])

# Filter out these slides
filtered_ann1 = {k: v for k, v in ann1.items() if k not in slides_to_remove}
filtered_ann2 = {k: v for k, v in ann2.items() if k not in slides_to_remove}

# Recompute errors
filtered_min_errors, filtered_max_errors = compute_errors(filtered_ann1, filtered_ann2)

print(f"After removing top 5 min and max difference slides ({len(slides_to_remove)} slides):")
print(f"Inter-annotator min error mean: {np.mean(filtered_min_errors):.5f}")
print(f"Inter-annotator max error mean: {np.mean(filtered_max_errors):.5f}")

After removing top 5 min and max difference slides (8 slides):
Inter-annotator min error mean: 0.03278
Inter-annotator max error mean: 0.06650
